# LangChain Chains — A Beginner's Guide

## What Are Chains?

In LangChain, a **chain** is a sequence of operations where data flows from one component to the next — much like Unix pipes (`cat file | grep "foo" | wc -l`) or pandas method chaining.

Chains let you:
- Connect prompts, language models, and output parsers together
- Process data through multiple steps sequentially
- Run multiple operations in parallel
- Route data based on conditions (if/else logic)

## The 3 Core Building Blocks

| Component | Role | Python Analogy |
|-----------|------|----------------|
| **PromptTemplate** | Builds the prompt string from variables | An f-string or `.format()` call |
| **Language Model** | Sends prompt to LLM, returns response | An API client like `requests.post()` |
| **Output Parser** | Converts raw LLM output to usable data | `json.loads()` or `dataclass(**data)` |

## LCEL — LangChain Expression Language

The pipe operator `|` connects components into a chain:
```python
chain = prompt | model | parser
```
Data flows left-to-right. This notebook covers **4 chain patterns** every LangChain developer should know.

---
## 1. Simple Chain

**Pattern**: `Prompt → Model → Parser`

This is the "Hello World" of LangChain. We'll build it step-by-step first (manual invoke at each stage), then assemble it with LCEL.

### How it works

```
"Python decorators"  →  PromptTemplate  →  ChatOpenAI  →  StrOutputParser  →  "1. Decorators are functions..."
   (user input)         (builds prompt)    (calls LLM)     (extracts text)        (final string)
```

We'll invoke each component manually first so you can see what each one returns. Then we'll glue them together with `|`.

#### Step 1 — Imports & environment

- `ChatOpenAI` — wrapper around OpenAI's chat completions API
- `PromptTemplate` — reusable prompt with `{placeholder}` variables
- `StrOutputParser` — turns `AIMessage` into a plain `str`
- `load_dotenv()` — reads `OPENAI_API_KEY` (and others) from a local `.env` file so you don't hardcode secrets

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

# Reads .env into os.environ — load_dotenv returns True on success
load_dotenv()

#### Step 2 — (Optional) Enable LangSmith tracing

LangSmith is LangChain's observability tool — like Sentry or Datadog, but for LLM calls. With tracing on, every chain run is logged with inputs, outputs, latency, and token cost.

- `LANGCHAIN_API_KEY` — auth (get one from [smith.langchain.com](https://smith.langchain.com))
- `LANGCHAIN_TRACING_V2="true"` — turns tracing on
- `LANGSMITH_PROJECT` — groups traces under a project name

> Skip this cell if you don't have a LangSmith account. The chains will still work.

In [ ]:

import os

os.environ["LANGCHAIN_API_KEY"] = str(os.getenv("LANGCHAIN_API_KEY"))
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_PROJECT"]="FinX-Agentic-AI"

#### Step 3 — Build the prompt

A `PromptTemplate` is essentially a parameterized f-string:

```python
prompt_template = PromptTemplate(
    template='Generate 5 interesting facts about {topic}',
    input_variables=['topic']
)
prompt_template.invoke({'topic': 'Python decorators'})
# → "Generate 5 interesting facts about Python decorators"
```

`.invoke()` substitutes the variables and returns a `PromptValue` object. Notice: we haven't called the LLM yet — this just builds the string.

In [ ]:
prompt_template = PromptTemplate(
    template='Generate 5 interesting facts about {topic}',
    input_variables=['topic']
)

prompt = prompt_template.invoke({'topic': 'Python decorators'})
print("Prompt:", prompt)

#### Step 4 — Call the model

`ChatOpenAI()` is the chat-model client. Defaults to `gpt-3.5-turbo`, but you can swap models for cost/quality tradeoffs:

```python
ChatOpenAI(model="gpt-4o-mini")    # cheap & fast, good default
ChatOpenAI(model="gpt-4o")         # best quality, more expensive
ChatOpenAI(temperature=0)          # deterministic output
```

`.invoke(prompt)` sends the prompt and returns an `AIMessage` object. Look at the printed output below — notice it's not just text. It's an object containing the response **plus** metadata (token counts, model name, finish reason). That metadata is what makes monitoring and cost tracking possible.

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini")
result = model.invoke(prompt)
print("Type:", type(result).__name__)
print("Content:\n", result.content)

#### Step 5 — Parse the output

The `AIMessage` object above is useful for inspection, but downstream code usually just wants the text. `StrOutputParser` does exactly that — `parser.invoke(ai_message)` returns `ai_message.content`.

Other parsers handle structured output:
- `JsonOutputParser` — parses JSON responses into dicts
- `PydanticOutputParser` — validates output against a Pydantic schema (we use this in the Conditional Chain section)
- `CommaSeparatedListOutputParser` — splits comma-separated lists into Python lists

In [ ]:
parser = StrOutputParser()
parsed_result = parser.invoke(result)

print("Type:", type(parsed_result).__name__)
print("Output:\n", parsed_result)

#### Step 6 — Compose with LCEL (the `|` operator)

Now the magic. Instead of calling `.invoke()` three times by hand, we chain the components with `|`:

```python
chain = prompt_template | model | parser
```

Read this left-to-right: the output of `prompt_template` flows into `model`, whose output flows into `parser`. Compare:

```python
# Without LCEL (verbose)
prompt_val = prompt_template.invoke({'topic': 'list comprehensions'})
ai_msg = model.invoke(prompt_val)
text = parser.invoke(ai_msg)

# With LCEL (one line)
text = chain.invoke({'topic': 'list comprehensions'})
```

Same result. The chain itself is also a `Runnable`, so it has `.invoke()`, `.batch()`, `.stream()`, and `.ainvoke()` (async) for free.

In [ ]:
# Assemble the chain with LCEL
chain = prompt_template | model | parser

# Run it end-to-end with a single call
result = chain.invoke({'topic': 'Python list comprehensions'})
print(result)

#### Step 7 — Visualize the chain

Every chain has a `.get_graph()` method that returns its computation graph. `.print_ascii()` draws it as ASCII art — super handy when chains get complex (the parallel and conditional examples below produce more interesting diagrams).

In [ ]:
chain.get_graph().print_ascii()

### 🧪 Try it yourself — Simple Chain

1. **Change the topic.** Run the chain with `'Django middleware'`, `'asyncio'`, or your own pick.
2. **Change the prompt.** Edit the template to `'Explain {topic} to a junior developer in 3 short paragraphs'`. Notice how only the prompt changes — model and parser stay the same.
3. **Change the model.** Swap `gpt-4o-mini` for `gpt-4o` and compare quality. Then try `ChatOpenAI(temperature=0)` and `temperature=1.5` — what happens?
4. **Batch it.** Call `chain.batch([{'topic': 'lambdas'}, {'topic': 'generators'}, {'topic': 'context managers'}])` to run three prompts in parallel with one call.

---
## 2. Sequential Chain

**Pattern**: `Prompt₁ → Model → Parser → Prompt₂ → Model → Parser`

The output of one chain feeds into the next. Think of it as a "pipeline of pipelines."

### Why chain LLM calls?

A single prompt asking "write a detailed tutorial AND summarize it in 5 points" produces worse results than two focused prompts. **Splitting reasoning into stages is one of the most reliable prompt-engineering tricks.**

```
{topic}  →  [Generate tutorial]  →  long_text  →  [Summarize to 5 points]  →  bullet_list
```

**Real-world use cases:**
- Draft → review → polish (content generation pipelines)
- Extract → classify → respond (customer support automation)
- Retrieve docs → synthesize answer (basic RAG)
- Code generation → test generation → documentation

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

#### Step 1 — Setup

We re-initialize the components here so each section is self-contained (you can run sections independently).

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

# Stage 1: generate a detailed tutorial about a programming topic
prompt1 = PromptTemplate(
    template='Write a detailed beginner-friendly tutorial on {topic}. Include code examples.',
    input_variables=['topic']
)

#### Step 2 — Define stage 1 (generator) and stage 2 (summarizer)

**Stage 1** takes a `topic` and produces a long tutorial.
**Stage 2** takes that tutorial and condenses it into 5 bullet points.

The key trick: stage 2's input variable is named `text`. We'll wire the output of stage 1 into that variable next.

In [ ]:
# Stage 2: condense the tutorial into 5 key takeaways
prompt2 = PromptTemplate(
    template='Summarize the following tutorial into exactly 5 bullet points a student can revise from:\n\n{text}',
    input_variables=['text']
)

#### Step 3 — Stitch the stages together

When the parser (after stage 1) emits a string, LCEL automatically passes it as `{text}` to `prompt2`. **This only works because the next prompt has exactly one input variable** — LangChain matches the single string to that lone variable. If `prompt2` had two variables, we'd need `RunnableParallel` or a custom mapping (covered in the next section).

In [ ]:
# Two pipelines glued together — one .invoke() runs both LLM calls in order
seq_chain = prompt1 | model | parser | prompt2 | model | parser

#### Step 4 — Run it

We provide the initial `topic` once. Everything else flows automatically:

1. `prompt1` builds: *"Write a detailed beginner-friendly tutorial on REST APIs..."*
2. `model` generates a long tutorial
3. `parser` extracts it as a string
4. That string becomes `{text}` for `prompt2`
5. `model` summarizes it into 5 bullets
6. `parser` returns the final string

Two LLM calls, one `.invoke()`.

In [ ]:
result = seq_chain.invoke({'topic': 'REST APIs in Python with Flask'})
print(result)

#### Inspect the graph

The ASCII diagram now shows two `ChatOpenAI` nodes — confirming the chain makes two LLM calls.

In [ ]:
seq_chain.get_graph().print_ascii()

### 🧪 Try it yourself — Sequential Chain

1. **Add a third stage.** Append `| prompt3 | model | parser` where `prompt3` turns the 5 bullets into a 5-question multiple-choice quiz. Chains can be arbitrarily long.
2. **Watch the cost.** Each `|` step that includes `model` is another OpenAI call. Run the chain once and check token usage in LangSmith.
3. **Break it on purpose.** Rename `prompt2`'s variable from `text` to `notes`. What error do you get, and why? (Hint: LangChain can't auto-wire when the name doesn't match.)
4. **Compare with a single prompt.** Write one big prompt that asks for both the tutorial AND the 5 bullets. Compare quality — which is sharper?

---
## 3. Parallel Chain

**Pattern**: Run multiple chains on the *same* input at the *same* time, then merge.

### Why parallel?

Imagine you're building a study tool. From a single textbook paragraph, you want **both** revision notes **and** a quiz. Running them sequentially wastes time — the two prompts are independent. `RunnableParallel` fires both LLM calls concurrently and waits for both to finish.

```
              ┌─→ notes_chain ──→ notes  ┐
{text} ──────┤                            ├──→ merge_chain ──→ final document
              └─→ quiz_chain  ──→ quiz   ┘
```

**Real-world use cases:**
- Generating multiple content formats from one source (blog + tweet + email)
- Multi-perspective analysis (pros + cons of a proposal)
- Multi-language translation in one pass
- Fan-out evaluation (run 3 different prompts and pick the best)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

#### Step 1 — Define two independent prompts

Both prompts take the **same input variable** (`text`) but produce different outputs:
- `prompt1` → short revision notes
- `prompt2` → 5 quick Q&A pairs

In [ ]:
prompt1 = PromptTemplate(
    template='Generate short, simple revision notes from the following text:\n\n{text}',
    input_variables=['text']
)

In [ ]:
prompt2 = PromptTemplate(
    template='Generate 5 short question-and-answer pairs from the following text:\n\n{text}',
    input_variables=['text']
)

#### Step 2 — Wrap each prompt in its own chain

Each is a complete `prompt | model | parser` pipeline. We could invoke them separately — but that would run them one after the other. Instead, we'll wrap them in `RunnableParallel` so they run concurrently.

In [ ]:
notes_chain = prompt1 | model | parser
quiz_chain = prompt2 | model | parser

#### Step 3 — `RunnableParallel`

`RunnableParallel` takes a dict mapping output keys to chains:

```python
parallel_chain = RunnableParallel({
    'notes': notes_chain,
    'quiz':  quiz_chain,
})
```

When you call `parallel_chain.invoke({'text': '...'})`:
- LangChain broadcasts the input to both chains
- Both LLM calls run **at the same time** (async under the hood)
- The result is a dict: `{'notes': '...', 'quiz': '...'}`

Note: the import path is `langchain_core.runnables` — we already imported it above.

In [ ]:
parallel_chain = RunnableParallel({
    'notes': notes_chain,
    'quiz':  quiz_chain,
})

#### Step 4 — Merge the parallel outputs

`RunnableParallel` emits a dict like `{'notes': '...', 'quiz': '...'}`. That dict matches the input variables of `prompt3` — LangChain auto-wires them by name. This is why naming your keys consistently matters.

In [ ]:
prompt3 = PromptTemplate(
    template=(
        'Combine the revision notes and quiz into a single neatly formatted study sheet.\n\n'
        '--- NOTES ---\n{notes}\n\n'
        '--- QUIZ ---\n{quiz}'
    ),
    input_variables=['notes', 'quiz']
)

#### Step 5 — Final chain: parallel + sequential combined

This is a hybrid:
- `parallel_chain` produces `{'notes': ..., 'quiz': ...}` (two LLM calls in parallel)
- `merge_chain` takes that dict and writes the final document (one more LLM call)

**Total LLM calls: 3** — but only 2 round-trips of latency, because notes and quiz happen simultaneously.

In [ ]:
merge_chain = prompt3 | model | parser

In [ ]:
final_chain = parallel_chain | merge_chain

#### Step 6 — Try it with real text

The paragraph below is about **Git branching** — a topic every Python developer touches daily. The chain will:
1. Extract revision notes from this paragraph (in parallel)
2. Generate 5 Q&A pairs from this paragraph (in parallel)
3. Merge both into a single study sheet

In [ ]:
text = """
A branch in Git is simply a movable pointer to a commit. The default branch in most repositories is called "main"
(formerly "master"). When you create a new branch with `git branch feature-x`, Git creates a new pointer at your
current commit — no files are copied, making the operation nearly instantaneous regardless of project size.

Branches let multiple developers work on different features in parallel without interfering with each other.
The typical workflow is: create a feature branch off main, commit changes there, push to the remote, open a
pull request, get reviewed, and merge back into main. This keeps main stable and deployable at all times.

Merging combines the work of two branches. Git supports two main strategies: a "fast-forward" merge (when main
has not advanced since the branch was created — Git simply moves the main pointer forward) and a "three-way"
merge (when both branches have new commits — Git creates a new merge commit that combines them). When the same
lines have been edited in both branches, Git cannot merge automatically and reports a merge conflict, which the
developer must resolve manually.

Rebasing is an alternative to merging. Instead of creating a merge commit, `git rebase main` replays your
branch's commits one by one on top of the latest main, producing a linear history. Rebasing rewrites commit
hashes, so the golden rule is: never rebase commits that have already been pushed and shared with others.
"""

In [ ]:
result = final_chain.invoke({'text':text})
print(result)

In [ ]:
final_chain.get_graph().print_ascii()

### 🧪 Try it yourself — Parallel Chain

1. **Add a third branch.** Inside `RunnableParallel`, add a `'flashcards': flashcards_chain` that generates Anki-style cards. Update `prompt3` to merge all three.
2. **Time it.** Wrap `final_chain.invoke(...)` in `time.time()` calls. Then run notes and quiz **sequentially** instead — what's the latency difference?
3. **Stop at parallel.** Skip the merge step and just call `parallel_chain.invoke({'text': text})`. Inspect the returned dict — this is often what you want for API responses.
4. **Different inputs per branch?** What if `notes_chain` needed `text` but `quiz_chain` needed `text` *and* a `difficulty` field? (Hint: look up `RunnablePassthrough` — it's the next layer of LCEL.)

---
## 4. Conditional Chain

**Pattern**: Classify the input first, then route to a different chain depending on the result.

### Why conditional?

Real applications rarely do the same thing for every input. A customer support bot replies differently to a complaint than to a thank-you note. A code-review bot reacts differently to a Python file than a YAML config. This pattern is essentially **`if/elif/else` for LLM chains** — implemented with `RunnableBranch`.

```
            ┌─ positive  → "Thank you!" prompt ┐
feedback ─→ Classifier ─┤─ negative  → Apology prompt        ├─→ response
            ├─ neutral   → Ask for details prompt│
            └─ escalate  → Escalation prompt    ┘
```

We use a **Pydantic schema** to force the classifier into one of four exact values (no free-form text) — that makes routing reliable.

**Real-world use cases:**
- Customer support triage (route by sentiment, urgency, or topic)
- Content moderation (different handlers for spam, abuse, off-topic)
- Multi-language responses (detect language, route to language-specific chain)
- Intent-based chatbots (greeting, FAQ, booking, complaint)

#### Step 1 — Imports

New pieces in this section:

| Import | What it does |
|--------|--------------|
| `ChatPromptTemplate` | Like `PromptTemplate` but supports message roles (system/human/assistant) — what real chat APIs expect |
| `RunnableBranch` | The if/elif/else router |
| `PydanticOutputParser` | Forces the LLM output to match a Pydantic schema (gives you a real Python object back) |
| `BaseModel`, `Field`, `Literal` | Standard Pydantic — defines the schema |

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableBranch
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv

load_dotenv()

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

#### Step 2 — Define the schema with Pydantic

This is the same Pydantic you use in FastAPI. `Literal[...]` restricts the field to exactly four values — if the LLM returns anything else, `PydanticOutputParser` raises a validation error. **That's the safety guarantee that makes routing reliable.**

```python
class Feedback(BaseModel):
    sentiment: Literal['positive', 'negative', 'neutral', 'escalate']
```

After parsing, we get a real `Feedback` object: `feedback.sentiment == 'positive'`.

In [ ]:
class Feedback(BaseModel):
    sentiment: Literal['positive', 'negative', 'neutral', 'escalate'] = Field(
        description='Sentiment of the customer feedback'
    )

feedback_parser = PydanticOutputParser(pydantic_object=Feedback)

#### Step 3 — Build the classifier prompt

`ChatPromptTemplate.from_messages([...])` lets you set multiple message roles:
- `"system"` — instructions/persona (not visible to the user)
- `"human"` — the actual question with `{placeholders}`

`.partial(format_instructions=...)` pre-fills one variable now and leaves the rest to be filled at invoke time. We use it to inject formatting hints generated by `feedback_parser.get_format_instructions()` — these tell the LLM exactly how to format its JSON response so the parser can validate it.

In [ ]:
classification_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a customer support classifier. Read each piece of feedback carefully."),
    ("human",
     "Classify the sentiment of this feedback as positive, negative, neutral, or escalate "
     "(use 'escalate' for legal/safety/severe issues that need a human):\n\n{feedback}\n\n{format_instructions}"),
]).partial(format_instructions=feedback_parser.get_format_instructions())

In [ ]:
# Test the classifier in isolation — returns a Feedback object, not a string
classification_chain = classification_prompt | model | feedback_parser

result = classification_chain.invoke({
    "feedback": "The product is okay, but I expected more features for the price."
})

print("Type:    ", type(result).__name__)
print("Sentiment:", result.sentiment)

#### Step 4 — One response prompt per branch

Each sentiment gets its own specialized prompt. In a real app, these might call different chains entirely (e.g., the `escalate` branch could trigger a Slack notification or create a Zendesk ticket — anything that's a `Runnable` works).

In [ ]:
positive_feedback_prompt = ChatPromptTemplate.from_messages([
    ("system", "You write warm, brief thank-you notes for happy customers."),
    ("human", "Write a thank-you reply (2-3 sentences) for this feedback: {feedback}"),
])

In [ ]:
negative_feedback_prompt = ChatPromptTemplate.from_messages([
    ("system", "You write empathetic apology replies and suggest a concrete next step."),
    ("human", "Write a reply (3-4 sentences) addressing this complaint: {feedback}"),
])

In [ ]:
neutral_feedback_prompt = ChatPromptTemplate.from_messages([
    ("system", "You politely ask the customer for more specifics to better understand their experience."),
    ("human", "Ask 2-3 follow-up questions in a single short reply for this feedback: {feedback}"),
])

In [ ]:
escalate_feedback_prompt = ChatPromptTemplate.from_messages([
    ("system", "You write internal handoff notes to escalate cases to a human support agent."),
    ("human", "Write a short escalation note for the on-call agent summarizing this feedback: {feedback}"),
])

#### Step 5 — Wire up the router with `RunnableBranch`

`RunnableBranch` works like a Python `if/elif/else`:

```python
RunnableBranch(
    (condition_1, chain_to_run_if_1),
    (condition_2, chain_to_run_if_2),
    ...
    default_chain,   # used when no condition matches
)
```

**The tricky part — passing data through:** by the time `RunnableBranch` runs, the input has been replaced by the classifier's `Feedback` object. But the response prompts still need the original `feedback` text. The fix is `RunnablePassthrough.assign`, which **keeps the original input and adds new fields alongside it**.

After `augment`, the data looks like:
```python
{'feedback': 'The product broke...', 'classification': Feedback(sentiment='negative')}
```

Now `RunnableBranch` can read `x['classification'].sentiment` to route, and the matched response prompt can still find `{feedback}`.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# Run classification, but keep the original 'feedback' field available downstream
augment = RunnablePassthrough.assign(classification=classification_chain)

# if/elif/else router
branches = RunnableBranch(
    (lambda x: x['classification'].sentiment == 'positive',
     positive_feedback_prompt | model | parser),
    (lambda x: x['classification'].sentiment == 'negative',
     negative_feedback_prompt | model | parser),
    (lambda x: x['classification'].sentiment == 'neutral',
     neutral_feedback_prompt | model | parser),
    # default branch (no condition needed) — runs when nothing else matched
    escalate_feedback_prompt | model | parser,
)

#### Step 6 — Assemble the full conditional chain

Two steps:
1. **`augment`** classifies and adds `classification` to the input dict
2. **`branches`** routes to the right response chain

In [ ]:
chain = augment | branches

In [ ]:
# Try different inputs to see different routes fire
examples = {
    "positive": "The product is excellent. I really enjoyed using it and found it very helpful.",
    "negative": "The product is terrible. It broke after just one use and the quality is very poor.",
    "neutral":  "The product is okay. It works as expected but nothing exceptional.",
    "escalate": "I'm allergic to one of the ingredients and ended up in the hospital. I want to speak to someone.",
}

# Pick a feedback to run — try changing this key!
key = "negative"
result = chain.invoke({"feedback": examples[key]})

print(f"=== Routed as: {key} ===")
print(result)

#### Inspect the graph

Notice the branch node — that's the router. The graph also shows that all four response chains exist in the graph, but only one runs per invocation.

In [ ]:
chain.get_graph().print_ascii()

### 🧪 Try it yourself — Conditional Chain

1. **Cycle through routes.** Run the cell above four times, switching `key` between `"positive"`, `"negative"`, `"neutral"`, and `"escalate"`. Confirm each generates a different reply style.
2. **Add a fifth category.** Extend `Literal` in `Feedback` to include `'feature_request'`. Add a matching `feature_request_prompt` and a new branch in `RunnableBranch`. (Hint: you'll also need to update the classifier's instructions.)
3. **Break the schema.** Change `Literal` to `Literal['great', 'bad']`. What happens when the LLM returns `"positive"`? (You'll see `PydanticOutputParser` raise a validation error — exactly the safety guarantee we wanted.)
4. **Mix in non-LLM branches.** Replace the `escalate` chain with a plain Python function: `RunnableLambda(lambda x: f"TICKET CREATED for: {x['feedback']}")`. Any `Runnable` works as a branch — LLM or not. This is how you bolt LangChain onto your existing Python services.

---
## Recap: When to use which chain?

| Pattern | Mental model | Good fit when... |
|---------|--------------|------------------|
| **Simple** `A \| B \| C` | Linear pipeline | One LLM call, one output |
| **Sequential** `... \| ... \| ...` | Multi-stage pipeline | Reasoning is sharper when split into steps (draft → refine → polish) |
| **Parallel** `RunnableParallel({...})` | Fan-out / fan-in | Multiple *independent* outputs from one input (notes + quiz, blog + tweet) |
| **Conditional** `RunnableBranch` | `if/elif/else` | Different inputs need different handling (triage, routing, multi-intent bots) |

## Concepts your students should walk away with

1. **Every component is a `Runnable`.** That's why `|`, `.invoke()`, `.batch()`, `.stream()`, `.ainvoke()` all work on every layer — prompts, models, parsers, parallel chains, branches.
2. **LCEL (`|`) ≈ Unix pipes.** Same intuition: small composable parts.
3. **Pydantic schemas turn LLM output into typed Python objects** — the bridge between fuzzy LLM text and reliable downstream code.
4. **`RunnablePassthrough.assign`** is the trick for "do something AND keep the original data" — used heavily in real RAG pipelines.
5. **`.get_graph().print_ascii()`** is a free debugger — use it whenever a chain misbehaves.

## What comes next (for follow-up lessons)

- **`RunnableLambda`** — wrap any Python function as a chain step
- **Memory** — give chains conversation history (`RunnableWithMessageHistory`)
- **Retrieval (RAG)** — combine retrievers with chains to ground LLM answers in your docs
- **Agents** — chains that decide what tool to call next
- **LangGraph** — when chains-as-DAGs aren't enough and you need real loops/state